In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
organizations_wikimedia_foundation_wikipedia_structured_contents_path = kagglehub.dataset_download('organizations/wikimedia-foundation/wikipedia-structured-contents')

print('Data source import complete.')

In [ ]:
import pandas as pd
from tqdm import tqdm
import json
import kagglehub
from datasets import Dataset
from huggingface_hub import notebook_login

In [ ]:
def read_data(file_path, max_records=None):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(tqdm(f)):
            if max_records and i >= max_records:
                break
            data.append(json.loads(line))
    return data

In [ ]:
file_path = kagglehub.dataset_download(
    "wikimedia-foundation/wikipedia-structured-contents",
    path="enwiki_namespace_0/enwiki_namespace_0_0.jsonl")
data = read_data(file_path)
df = pd.DataFrame(data)
del data

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.drop(["date_created", "additional_entities"], axis=1, inplace=True)
df.dropna(inplace=True)
df = df.reset_index()
df.pop("index")
df.head()

In [ ]:
def get_value_safe(section):
    try:
        return section[0]["has_parts"][0]["value"]
    except (IndexError, KeyError, TypeError):
        return None

In [ ]:
def cleaning_treatment(df):
    df["in_language"] = "en"
    df["main_entity"] = [df["main_entity"][i]["url"] for i in range(len(df))]
    df["version"] = [v["comment"] if isinstance(v, dict) and "comment" in v else None for v in df["version"]]
    df["is_part_of"] = [df["is_part_of"][i]["url"] for i in range(len(df))]
    df["license"] = [df["license"][i][0]["name"] for i in range(len(df))]
    df["sections"] = [get_value_safe(s) for s in df["sections"]]
    df["image"] = [df["image"][i]["content_url"] for i in range(len(df))]
    df["event"] = [df["event"][i]["type"] for i in range(len(df))]
    df["date_modified"] = pd.to_datetime(df["date_modified"])

    return df

In [ ]:
df = cleaning_treatment(df)

In [ ]:
df.head()

In [ ]:
notebook_login()

In [ ]:
hf_dataset = Dataset.from_pandas(df)
hf_dataset.push_to_hub("Ambrosio1994/wikipedia_structured_contents")